In [1]:
import os
import re
import unicodedata
import glob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from stop_words_custom import stop_word_list
from title_map import title_map
from unpack_documents import (load_documents, 
                              preprocess, 
                              CustomPorterStemmer,
                              clean_titles_from_csv,
                              clean_titles_from_filenames)

# Directory containing .txt files
TEXT_DIR = "text" 

# Directory containing other metadata
DATA_DIR = "data"

In [2]:
male_pronouns = ["he", "him"]
female_pronouns = ["she", "her"]

## Load documents

In [3]:
filepaths = glob.glob(os.path.join(TEXT_DIR, "*.txt"))

corpus, filenames = load_documents(filepaths)

Loaded 762 documents


## Preprocessing documents (stemming)

In [4]:
stemmer = CustomPorterStemmer()

corpus_stemmed = [preprocess(doc, stemmer) for doc in corpus]

## Bag of words

In [5]:
vectorizer = CountVectorizer(
    strip_accents='unicode',
    stop_words=stop_word_list,
    max_df=1.0,
    min_df=5
)

X = vectorizer.fit_transform(corpus_stemmed)

print("Matrix shape:", X.shape)

/Users/mjl09005/Library/CloudStorage/OneDrive-UniversityofConnecticut/gvp/docx_env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['articl', 'mon'] not in stop_words.
  warnings.warn(


Matrix shape: (762, 14355)


In [6]:
feature_names = vectorizer.get_feature_names_out()

titles = clean_titles_from_filenames(filenames)

bow_df = pd.DataFrame(X.toarray(), columns=feature_names, index=titles)

bow_df.head()

,aa,aaron,aback,abalon,abandon,abash,abat,abattoir,abbey,abbi,...,zigzag,zillion,zinc,zip,zipper,zombi,zone,zoo,zoom,zu
brisk money,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
specimen 313,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
when we were heroes,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
tie a yellow ribbon,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
apologue,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [7]:
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

word_counts.head()

he      70200
him     69431
she     59722
her     58112
they    29427
dtype: int64

## Totals across all documents

In [8]:
sum_male = 0
sum_female = 0

for pronoun in male_pronouns:
    sum_male += word_counts.get(pronoun, 0)

for pronoun in female_pronouns:
    sum_female += word_counts.get(pronoun, 0)

print(f"Total male pronouns: {sum_male}")
print(f"Total female pronouns: {sum_female}")

Total male pronouns: 139631
Total female pronouns: 117834


In [9]:
bow_df["male_total"] = bow_df[male_pronouns].sum(axis=1)
bow_df["female_total"] = bow_df[female_pronouns].sum(axis=1)

## Load compiled stats to see totals by document

In [10]:
stats = pd.read_csv('data/compiled_stats.csv')

stats.head()

,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count
0,20-Jul-08,2008,The Leviathan,Wesley Allsbrook,She/her,NaN,NaN,NaN,Comics,NaN,NaN,NaN,NaN,14
1,20-Jul-08,2008,Down on the Farm,Charles Stross,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Fantasy,Lovecraftian,NaN,12682.0,NaN
2,20-Jul-08,2008,After the Coup,John Scalzi,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Science Fiction,Space Opera,NaN,7087.0,NaN
3,5-Aug-08,2008,Old Flames Flare for Sweetie 'n Me,Joel Priddy,He/him,NaN,NaN,NaN,Comics,NaN,NaN,NaN,NaN,13
4,6-Aug-08,2008,The Things That Make Me Weak and Strange Get E...,Cory Doctorow,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,NaN,NaN,NaN,15708.0,NaN


### Needs cleaning to get titles to match

In [11]:
stats = clean_titles_from_csv(stats)

stats['cleaned_title'] = (
    stats['cleaned_title']
    .replace(title_map)
)

In [37]:
pronoun_data_left = (
    stats[(stats['Type']=="Original Fiction") & (stats['Category']!="Reprint")].merge(
        bow_df[['male_total', 'female_total']],
        left_on='cleaned_title',
        right_index=True,
        how='left'
    )
    .drop_duplicates()
)

pronoun_data_right = (
    stats[(stats['Type']=="Original Fiction") & (stats['Category']!="Reprint")].merge(
        bow_df[['male_total', 'female_total']],
        left_on='cleaned_title',
        right_index=True,
        how='right'
    )
    .drop_duplicates()
)

### Sanity check: dataset counts

In [38]:
print("from csv:", len(stats[(stats['Type']=="Original Fiction") & (stats['Category']!="Reprint")]))
print("from documents:", len(bow_df))

print("from left merge:", len(pronoun_data_left))
print("from right merge:", len(pronoun_data_right))

from csv: 656
from documents: 762
from left merge: 657
from right merge: 772


In [39]:
duplicates = pronoun_data_left[pronoun_data_left.duplicated(subset=['Title', 'Author'], keep=False)]
print("Duplicates in left merge:", len(duplicates))

Duplicates in left merge: 2


In [40]:
duplicates

,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count,cleaned_title,male_total,female_total
379,2-Apr-14,2014,The Devil in America,Kai Ashante Wilson,He/him,Ann VanderMeer,NaN,NaN,Original Fiction,Fantasy,Historical,NaN,15086.0,NaN,the devil in america,248.0,362.0
379,2-Apr-14,2014,The Devil in America,Kai Ashante Wilson,He/him,Ann VanderMeer,NaN,NaN,Original Fiction,Fantasy,Historical,NaN,15086.0,NaN,the devil in america,249.0,362.0


### Look at unmatched rows from each side

In [41]:
num_nans_left = pronoun_data_left['female_total'].isna().sum()
print(num_nans_left)
unmatched_left = pronoun_data_left[pronoun_data_left['female_total'].isna()]
unmatched_left

1


,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count,cleaned_title,male_total,female_total
52,4-Dec-09,2009,I Speak Fluent Giraffe,Jason Henninger,He/him,NaN,NaN,NaN,Original Fiction,Humor,NaN,NaN,0.0,NaN,i speak fluent giraffe,NaN,NaN


In [42]:
num_nans_right = pronoun_data_right['Title'].isna().sum()
print(num_nans_right)
unmatched_right = pronoun_data_right[pronoun_data_right['Title'].isna()]
unmatched_right

116


,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count,cleaned_title,male_total,female_total
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,specimen 313,174,52
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apologue,20,2
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tortured,311,151
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fields of gold,310,276
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,two from weird tales,69,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,the fox wife,3,0
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fangs for hire,329,38
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,my garden,2,0
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,some desperado,201,498
